# CampaignOptima: Phase 1 — Exploratory Data Analysis & Behavioral Feature Engineering

**Author:** CampaignOptima Decision Science Team  
**Objective:** Transform raw bank marketing contact data into customer behavioral intelligence, uncover non-linear conversion dynamics, analyze severe class imbalance (~11.5%), and construct domain-specific feature representations.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in system path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.data_loader import BankMarketingDataLoader
from src.features import BehavioralFeatureEngineer, CampaignOptimaFeaturePipeline

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

## 1. Data Ingestion & Delimiter Handling (`;` separator)
The dataset originates from direct telemarketing campaigns of a Portuguese banking institution (UCI Bank Marketing). It employs a semicolon (`;`) delimiter.

In [ ]:
loader = BankMarketingDataLoader()
df_raw = loader.load_raw_data()
print(f"Raw dataset dimensions: {df_raw.shape}")
df_raw.head()

## 2. Target Variable & Severe Class Imbalance Inspection
Term deposit conversion (`y`) exhibits severe positive skew (~11% positive conversion rate).

In [ ]:
y_dist = df_raw["y"].value_counts(normalize=True) * 100
print("Conversion Distribution (%):")
print(y_dist)

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df_raw, x="y", palette=["#EF4444", "#10B981"], ax=ax)
ax.set_title(f"Severe Class Imbalance: {y_dist['yes']:.1f}% Conversion Rate")
ax.set_ylabel("Customer Count")
ax.set_xlabel("Subscribed to Term Deposit (y)")
plt.show()

## 3. Demographics & Financial Health vs Conversion
Exploring how professional occupation, education, and account balances correlate with subscription likelihood.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Job Conversion Rate
job_conv = df_raw.groupby("job")["y"].apply(lambda s: (s == "yes").mean() * 100).sort_values(ascending=False)
sns.barplot(x=job_conv.values, y=job_conv.index, palette="viridis", ax=axes[0])
axes[0].set_title("Conversion Rate by Profession (%)")
axes[0].set_xlabel("Conversion Rate (%)")

# Housing Loan vs Balance Distribution
sns.boxplot(data=df_raw, x="housing", y="balance", hue="y", showfliers=False, palette="Set2", ax=axes[1])
axes[1].set_title("Account Balance by Housing Mortgage & Conversion")
axes[1].set_ylabel("Bank Balance ($)")
plt.tight_layout()
plt.show()

## 4. Behavioral Feature Engineering Validation
We now engineer our domain-specific features:
1. `Campaign_Intensity_Ratio` = $\frac{\text{campaign}}{\text{previous} + 1.0}$
2. `Recency_Weight_Score` = $\exp(-\lambda \cdot \text{pdays}) \cdot \log(1 + \text{previous})$
3. `Financial_Stability_Index` = $\text{Balance}_{\text{norm}} - 0.35 \cdot \text{Housing} - 0.40 \cdot \text{Loan} - 0.50 \cdot \text{Default}$

In [ ]:
engineer = BehavioralFeatureEngineer()
df_engineered = engineer.transform(df_raw)

print("Engineered Features Summary:")
df_engineered[["Campaign_Intensity_Ratio", "Recency_Weight_Score", "Financial_Stability_Index", "log_duration"]].describe()

## 5. Contact Fatigue & Burnout Analysis
Testing the hypothesis that excessive repeated calls diminish customer conversion probability.

In [ ]:
burnout_stat = df_engineered.groupby(np.clip(df_engineered["campaign"], 1, 10))["y"].apply(
    lambda s: (s == "yes").mean() * 100
).reset_index()
burnout_stat.columns = ["Contacts_Placed", "Conversion_Rate"]

plt.figure(figsize=(8, 4))
plt.plot(burnout_stat["Contacts_Placed"], burnout_stat["Conversion_Rate"], marker="o", color="#DC2626", lw=2.5)
plt.title("Telemarketing Fatigue: Sharp Drop-off After 3 Calls")
plt.xlabel("Contacts Placed in Campaign")
plt.ylabel("Conversion Rate (%)")
plt.axvspan(3.5, 10, color="red", alpha=0.1, label="Burnout Zone")
plt.legend()
plt.show()